# Sistem Prediksi Saham RNN: Multi-Hari (H+1 s.d H+31) & Scalping Intraday (5 Menit)
### Sektor BEI: Perbankan, Makanan & Minuman, Perkebunan, dan Mineral & Logam

Notebook ini dirancang untuk **penggunaan harian (Everyday Use)** investor & trader bursa Indonesia (BEI/IDX).

#### Fitur Unggulan Proyek:
1. **Dual Environment**: Mendukung **Google Colab** (dengan auto-mount Google Drive) dan **Local Machine** (VS Code / Jupyter Notebook / Terminal lokal).
2. **Cakupan Sektor Lengkap (165+ Emiten)**: Memuat seluruh emiten resmi BEI untuk sektor Perbankan (47), Makanan & Minuman (52), Perkebunan (34), dan Mineral & Logam (34).
3. **Fleksibilitas Operasional Harian**:
   - `MODE_RUN = "TOP_LIQUID"`: Scan kilat ~2-3 menit untuk saham-saham paling likuid.
   - `MODE_RUN = "CUSTOM"`: Analisis cepat untuk watchlist saham pilihan Anda hari ini.
   - `MODE_RUN = "SCAN_ALL"`: Pemindaian komprehensif seluruh emiten dalam sektor.
   - `RETRAIN_MODEL = False`: Fitur **Model Caching** untuk prediksi instan dalam hitungan detik tanpa perlu melatih ulang neural network setiap hari.
4. **Multi-Horizon Swing Trading**: Proyeksi harga bursa ke **H+1, H+7, H+14, H+21, H+30, dan H+31**.
5. **Scalping Intraday 5 Menit**: Proyeksi candle **+5m, +15m, +30m, dan +60m** lengkap dengan kalkulasi komisi bursa (Fee Breakeven) & deteksi lonjakan volume.
6. **Actionable Trading Plan**: Setiap emiten dilengkapi sinyal (**STRONG BUY / BUY / WAIT / SELL**), Target Price (TP), Stop Loss (SL), dan Risk-to-Reward Ratio (RRR).
7. **Visualisasi Dinamis & Auto-Save**: Menghasilkan grafik proyeksi beresolusi tinggi yang otomatis tersimpan ke folder laporan.

In [ ]:
import os
import sys

# =====================================================================
# 1. DETEKSI OTOMATIS LINGKUNGAN (GOOGLE COLAB vs LOKAL PC)
# =====================================================================
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("[LINGKUNGAN] Berjalan di Google Colab. Menghubungkan Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/Stock_RNN_Project'
else:
    print("[LINGKUNGAN] Berjalan di Mesin Lokal (Local Machine / VSCode / Jupyter).")
    # Menggunakan direktori proyek lokal saat ini
    BASE_DIR = os.path.abspath('./Stock_RNN_Project')

# Struktur folder proyek terorganisir
DATA_DAILY_DIR = os.path.join(BASE_DIR, 'data_daily')
DATA_SCALP_DIR = os.path.join(BASE_DIR, 'data_scalping')
MODEL_DIR = os.path.join(BASE_DIR, 'models')
LOG_DIR = os.path.join(BASE_DIR, 'logs')
REPORT_DIR = os.path.join(BASE_DIR, 'reports')

for folder in [DATA_DAILY_DIR, DATA_SCALP_DIR, MODEL_DIR, LOG_DIR, REPORT_DIR]:
    os.makedirs(folder, exist_ok=True)

print("=" * 65)
print(f"Direktori Basis : {BASE_DIR}")
print(f"Mode Eksekusi   : {'Google Colab (Drive Persistent)' if IN_COLAB else 'Local Workspace'}")
print("Semua direktori data, model, log, dan laporan siap!")
print("=" * 65)

# 2. Instalasi Library, Daftar Lengkap Saham Per Sektor & Pengaturan Harian
Di cell ini terdapat **daftar ekstensif emiten BEI** untuk 4 sektor utama (165+ emiten),
serta pengaturan operasional harian yang sangat fleksibel.

In [ ]:
# Instalasi / update library yfinance jika diperlukan
try:
    import yfinance as yf
except ImportError:
    !pip install -q --upgrade yfinance
    import yfinance as yf

import gc
import time
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Kunci random seed untuk reproduksibilitas hasil eksperimen
np.random.seed(42)
tf.random.set_seed(42)

# =====================================================================
# DAFTAR LENGKAP EMITEN BURSA EFEK INDONESIA (BEI) PER SEKTOR (167 EMITEN)
# =====================================================================
SEKTOR_SAHAM = {
    "Perbankan": [
        # 1. Bank Besar / KBMI 4 & Syariah Unggulan
        "BBCA", "BBRI", "BMRI", "BBNI", "BRIS", "BBTN", "BNGA", "BDMN",
        "NISP", "PNBN", "BNLI", "BNII", "MEGA", "BTPN", "BTPS",
        # 2. Bank Pembangunan Daerah (BPD) & Bank Digital
        "BJBR", "BJTM", "BEKS", "ARTO", "BBYB", "BBHI", "AGRO", "BANK",
        "AMAR", "BBSI",
        # 3. Bank Swasta Nasional, Komersial & Khusus
        "BACA", "BABP", "BBKP", "BCIC", "BGTG", "BINA", "BKSW", "BMAS",
        "BNBA", "BBMD", "BSIM", "BSWD", "BVIC", "DNAR", "INPC", "MASB",
        "MAYA", "MCOR", "NOBU", "PNBS", "SDRA", "AGRS"
    ],
    "Makanan": [
        # 1. Makanan Olahan Kemasan, Biskuit, Roti & Bumbu
        "ICBP", "INDF", "MYOR", "ROTI", "GOOD", "STTP", "SKLT", "SKBM",
        "CEKA", "CAMP", "KEJU", "HOKI", "BUDI", "AISA", "ALTO", "COCO",
        "FOOD", "IKAN", "PSDN", "PMMP", "ENZO", "BOBA", "NASI", "TAYS",
        "GULA", "TRGU", "WAPO",
        # 2. Minuman Olahan, Air Mineral, Teh, Bir & Wine
        "CMRY", "ULTJ", "CLEO", "ADES", "MLBI", "DLTA", "BEER", "WINE",
        "DMND", "SOUL", "TGUK",
        # 3. Peternakan Unggas, Olahan Daging & Perikanan (Food Producers)
        "CPIN", "JPFA", "MAIN", "WMUU", "BEEF", "CPRO", "DSFI", "CRAB",
        "PCAR", "SIPD", "AMMS", "DEWI", "ISEA", "AGAR"
    ],
    "Perkebunan": [
        # 1. Kelapa Sawit (CPO) Blue Chip & Konglomerasi Agribisnis
        "AALI", "LSIP", "TAPG", "DSNG", "SSMS", "SIMP", "SGRO", "STAA",
        "TBLA", "SMAR",
        # 2. Produsen CPO Menengah, Kilang Hilir / Refinery & Bioenergi
        "NSSS", "BWPT", "GZCO", "JAWA", "ANJT", "CSRA", "FAPA", "MGRO",
        "PGUN", "PNGO", "PSGO", "UNSP", "JARR", "CBUT", "MKTR", "TLDN",
        "OILS", "PTPS",
        # 3. Agribisnis, Benih, Perkebunan Lain & Komoditas Pertanian
        "PALM", "GOLL", "MAGP", "BISI", "ANDI", "FISH"
    ],
    "Mineral": [
        # 1. Emas, Tembaga, Nikel, Timah, Bauksit & Logam Mulia
        "AMMN", "ANTM", "INCO", "MDKA", "MBMA", "NCKL", "TINS", "BRMS",
        "PSAB", "ARCI", "NICL", "NICE", "DKFT", "IFSH", "CITA", "ZINC",
        "SQMI", "SMGA", "HRUM", "CUAN",
        # 2. Smelter Logam Dasar, Baja, Tembaga, Aluminium & Fabrikasi
        "KRAS", "ISSP", "GGRP", "INAI", "ALKA", "ALMI", "GDST", "BAJA",
        "NIKL", "LION", "TBMS", "KBLI", "VOKS", "IKBI"
    ]
}

# =====================================================================
# PENGATURAN OPERASIONAL HARIAN (EVERYDAY USE CONTROLS)
# =====================================================================

# 1. Pilihan Sektor yang Ingin Dianalisis:
#    ["ALL"] untuk semua sektor, atau spesifik misal ["Perbankan", "Mineral"]
PILIH_SEKTOR = ["ALL"]

# 2. Mode Eksekusi:
#    - "TOP_LIQUID" : (Sangat Disarankan untuk Harian) Scan cepat saham-saham paling likuid & berkapitalisasi solid (~25 emiten total).
#    - "CUSTOM"     : Jalankan daftar saham spesifik yang Anda masukkan di DAFTAR_CUSTOM.
#    - "SCAN_ALL"   : Scan seluruh 167 emiten (memerlukan waktu lebih lama).
MODE_RUN = "TOP_LIQUID"

# 3. Watchlist Custom (Aktif jika MODE_RUN = "CUSTOM"):
DAFTAR_CUSTOM = ["BBCA", "BBRI", "BMRI", "ANTM", "MDKA", "AMMN", "ICBP", "MYOR", "AALI", "LSIP"]

# 4. Saham Paling Likuid / Core Watchlist per Sektor (Untuk MODE_RUN = "TOP_LIQUID"):
TOP_LIQUID_SAHAM = {
    "Perbankan": ["BBCA", "BBRI", "BMRI", "BBNI", "BRIS", "BBTN"],
    "Makanan":   ["ICBP", "INDF", "MYOR", "CMRY", "CPIN", "JPFA"],
    "Perkebunan":["AALI", "LSIP", "TAPG", "DSNG", "SSMS", "TBLA"],
    "Mineral":   ["ANTM", "AMMN", "MDKA", "INCO", "NCKL", "TINS", "BRMS"]
}

# 5. Retrain Model vs Model Caching (Inferensi Cepat Harian):
#    - True  : Melatih ulang neural network RNN dan memperbarui bobot model.
#    - False : Menggunakan model yang sudah tersimpan di folder models/ untuk prediksi instan dalam hitungan detik!
RETRAIN_MODEL = True

# 6. Target Prediksi Multi-Hari (H+1 s.d. H+31 hari bursa):
MAX_HORIZON_HARI = 31
TARGET_HARI = [1, 7, 14, 21, 30, 31]

# 7. Target Prediksi Scalping Intraday (5 Menit):
SCALP_INTERVAL = "5m"
MAX_HORIZON_SCALP = 12             # 12 candle 5m = 60 menit ke depan
TARGET_SCALP_INDEX = [0, 2, 5, 11] # Titik prediksi: +5m, +15m, +30m, +60m
FEE_THRESHOLD_PERSEN = 0.40        # Estimasi total fee transaksi beli (0.15%) + jual (0.25%)

# 8. Hyperparameter Neural Network RNN:
WINDOW_SIZE = 30                   # Jumlah data historis mundur (30 hari / 30 candle)
EPOCHS = 30                        # Jumlah iterasi training
BATCH_SIZE_DAILY = 32
BATCH_SIZE_SCALP = 64
DELAY_DOWNLOAD_DETIK = 0.35        # Jeda antar request Yahoo Finance agar aman dari rate-limit

# =====================================================================
# PEMBENTUKAN DAFTAR SAHAM AKTIF BERDASARKAN MODE_RUN
# =====================================================================
DAFTAR_SAHAM = {}
sektor_aktif = list(SEKTOR_SAHAM.keys()) if "ALL" in PILIH_SEKTOR else PILIH_SEKTOR

if MODE_RUN == "CUSTOM":
    for ticker in DAFTAR_CUSTOM:
        sektor_temu = "Custom"
        for s, emiten_list in SEKTOR_SAHAM.items():
            if ticker in emiten_list:
                sektor_temu = s
                break
        DAFTAR_SAHAM[f"{sektor_temu}_{ticker}"] = f"{ticker}.JK"

elif MODE_RUN == "TOP_LIQUID":
    for s in sektor_aktif:
        for ticker in TOP_LIQUID_SAHAM.get(s, []):
            DAFTAR_SAHAM[f"{s}_{ticker}"] = f"{ticker}.JK"

else:  # SCAN_ALL
    for s in sektor_aktif:
        for ticker in SEKTOR_SAHAM.get(s, []):
            DAFTAR_SAHAM[f"{s}_{ticker}"] = f"{ticker}.JK"

print("=" * 65)
print(f"Mode Operasional Aktif : {MODE_RUN}")
print(f"Status Retrain Model   : {'Retrain Bobot Baru' if RETRAIN_MODEL else 'Gunakan Model Tersimpan (Cepat)'}")
print(f"Total Saham Siap Run   : {len(DAFTAR_SAHAM)} emiten")
print("Rincian per sektor aktif:")
for s in sektor_aktif:
    jml_sektor = 0
    for k in DAFTAR_SAHAM.keys():
        if k.startswith(s + '_'):
            jml_sektor += 1
    print(f"  - {s:12s}: {jml_sektor} emiten")
print("=" * 65)

# 3. Engine Unduh & Update Data yfinance (Harian & Intraday 5 Menit)
Mendukung **Incremental Update** (hanya mengunduh data terbaru jika file CSV sudah ada)
serta pemfilteran otomatis saham yang suspensi, delisted, atau tidak likuid.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

def bersihkan_dan_validasi(df, nama, is_daily=True):
    if df is None or df.empty:
        return pd.DataFrame()

    # Ratakan multi-index jika ada
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    kolom_wajib = ['Open', 'High', 'Low', 'Close', 'Volume']
    if not all(k in df.columns for k in kolom_wajib):
        return pd.DataFrame()

    df = df[kolom_wajib].copy()

    # Standarisasi index waktu ke format WIB tanpa informasi timezone kompleks
    df.index = pd.to_datetime(df.index)
    if df.index.tz is not None:
        df.index = df.index.tz_convert('Asia/Jakarta').tz_localize(None)

    df = df[~df.index.duplicated(keep='last')].sort_index()
    df = df.dropna(subset=['Close'])
    df = df[df['Close'] > 0]

    return df


def update_data_harian(daftar_saham, periode_awal="3y"):
    print(f"\n=== [1] Mengunduh & Memperbarui Data Harian ({len(daftar_saham)} Emiten) ===")
    total = len(daftar_saham)
    sukses, gagal = 0, []

    for idx, (nama, ticker) in enumerate(daftar_saham.items(), start=1):
        file_path = os.path.join(DATA_DAILY_DIR, f"{nama}_1d.csv")
        try:
            if os.path.exists(file_path):
                df_lama = pd.read_csv(file_path, index_col=0, parse_dates=True)
                tgl_terakhir = df_lama.index.max()
                start_fetch = (tgl_terakhir - pd.Timedelta(days=7)).strftime('%Y-%m-%d')

                df_baru = yf.download(ticker, start=start_fetch, interval="1d", auto_adjust=True, progress=False)
                df_baru = bersihkan_dan_validasi(df_baru, nama, is_daily=True)

                perlu_refresh = False
                if not df_baru.empty:
                    overlap = df_lama.index.intersection(df_baru.index)
                    if len(overlap) > 0:
                        selisih = np.abs(df_lama.loc[overlap, 'Close'] - df_baru.loc[overlap, 'Close']) / df_lama.loc[overlap, 'Close']
                        if (selisih > 0.005).any():
                            perlu_refresh = True

                if perlu_refresh:
                    df_full = yf.download(ticker, period=periode_awal, interval="1d", auto_adjust=True, progress=False)
                    df_final = bersihkan_dan_validasi(df_full, nama, is_daily=True)
                else:
                    df_final = pd.concat([df_lama, df_baru])
                    df_final = df_final[~df_final.index.duplicated(keep='last')].sort_index()
            else:
                df_full = yf.download(ticker, period=periode_awal, interval="1d", auto_adjust=True, progress=False)
                df_final = bersihkan_dan_validasi(df_full, nama, is_daily=True)

            if not df_final.empty and len(df_final) >= (WINDOW_SIZE + MAX_HORIZON_HARI + 10):
                df_final.to_csv(file_path)
                sukses += 1
                print(f"  [{idx:02d}/{total:02d}] OK {nama:20s} | {len(df_final)} baris | Last: {df_final.index[-1].strftime('%Y-%m-%d')}")
            else:
                gagal.append((nama, "Data terlalu sedikit / emiten tidak aktif"))
        except Exception as e:
            gagal.append((nama, str(e)))

        time.sleep(DELAY_DOWNLOAD_DETIK)

    print(f"Selesai Data Harian: {sukses} berhasil, {len(gagal)} gagal/di-skip.")
    return sukses, gagal


def update_data_scalping(daftar_saham, periode="60d", interval="5m"):
    print(f"\n=== [2] Mengunduh & Memperbarui Data Intraday {interval} ({len(daftar_saham)} Emiten) ===")
    total = len(daftar_saham)
    sukses, gagal = 0, []

    for idx, (nama, ticker) in enumerate(daftar_saham.items(), start=1):
        file_path = os.path.join(DATA_SCALP_DIR, f"{nama}_{interval}.csv")
        try:
            df_baru = yf.download(ticker, period=periode, interval=interval, auto_adjust=True, progress=False)
            df_baru = bersihkan_dan_validasi(df_baru, nama, is_daily=False)

            if os.path.exists(file_path) and not df_baru.empty:
                df_lama = pd.read_csv(file_path, index_col=0, parse_dates=True)
                df_final = pd.concat([df_lama, df_baru])
                df_final = df_final[~df_final.index.duplicated(keep='last')].sort_index()
            else:
                df_final = df_baru

            if not df_final.empty and len(df_final) >= (WINDOW_SIZE + MAX_HORIZON_SCALP + 10):
                df_final.to_csv(file_path)
                sukses += 1
                print(f"  [{idx:02d}/{total:02d}] OK {nama:20s} | {len(df_final)} candle | Last: {df_final.index[-1].strftime('%Y-%m-%d %H:%M')}")
            else:
                gagal.append((nama, "Data intraday kosong / emiten suspensi"))
        except Exception as e:
            gagal.append((nama, str(e)))

        time.sleep(DELAY_DOWNLOAD_DETIK)

    print(f"Selesai Data Scalping: {sukses} berhasil, {len(gagal)} gagal/di-skip.")
    return sukses, gagal

# Jalankan sinkronisasi data
sukses_daily, gagal_daily = update_data_harian(DAFTAR_SAHAM)
sukses_scalp, gagal_scalp = update_data_scalping(DAFTAR_SAHAM, periode="60d", interval=SCALP_INTERVAL)

# 4. Preprocessing Sliding Window & Arsitektur RNN Multi-Output
Membangun dataset sekuensial multi-langkah dan arsitektur Recurrent Neural Network (RNN)
yang mampu memproyeksikan seluruh vektor horizon secara simultan.

In [ ]:
def buat_sliding_window_multi(data_scaled, window_size, horizon):
    X, y = [], []
    for i in range(window_size, len(data_scaled) - horizon + 1):
        X.append(data_scaled[i - window_size : i, 0])
        y.append(data_scaled[i : i + horizon, 0])
    X = np.array(X)
    y = np.array(y)
    return X.reshape((X.shape[0], X.shape[1], 1)), y


def siapkan_dataset(file_csv, window_size, horizon, rasio_train=0.8):
    if not os.path.exists(file_csv):
        raise FileNotFoundError(f"File {file_csv} tidak ditemukan.")

    df = pd.read_csv(file_csv, index_col=0, parse_dates=True)
    harga_close = df[['Close']].dropna().values.astype(float)

    min_baris = window_size + horizon + 25
    if len(harga_close) < min_baris:
        raise ValueError(f"Baris data ({len(harga_close)}) kurang dari kebutuhan minimum ({min_baris}).")

    # Deteksi saham stagnan / suspensi (std = 0)
    if np.std(harga_close) < 1e-5:
        raise ValueError("Harga saham stagnan (std dev mendekati 0 / kemungkinan suspensi).")

    train_size = int(len(harga_close) * rasio_train)
    train_raw = harga_close[:train_size]
    test_raw = harga_close[train_size - window_size:]

    scaler = MinMaxScaler(feature_range=(0, 1))
    train_scaled = scaler.fit_transform(train_raw)
    test_scaled = scaler.transform(test_raw)

    X_train, y_train = buat_sliding_window_multi(train_scaled, window_size, horizon)
    X_test, y_test = buat_sliding_window_multi(test_scaled, window_size, horizon)

    if len(X_train) < 10 or len(X_test) < 3:
        raise ValueError(f"Jumlah window sampel tidak mencukupi (train={len(X_train)}, test={len(X_test)}).")

    # Window terakhir dari data aktual untuk memproyeksikan masa depan
    last_window = scaler.transform(harga_close[-window_size:])
    X_future = last_window.reshape((1, window_size, 1))

    # Hitung rata-rata volume terakhir jika kolom Volume ada
    vol_terakhir = float(df['Volume'].iloc[-1]) if 'Volume' in df.columns else 0.0
    vol_ma20 = float(df['Volume'].rolling(20).mean().iloc[-1]) if 'Volume' in df.columns else 1.0
    rasio_vol = (vol_terakhir / vol_ma20) if vol_ma20 > 0 else 1.0

    return {
        "X_train": X_train, "y_train": y_train,
        "X_test": X_test, "y_test": y_test,
        "X_future": X_future,
        "harga_terakhir": float(harga_close[-1, 0]),
        "waktu_terakhir": str(df.index[-1]),
        "scaler": scaler,
        "rasio_vol": rasio_vol
    }


def bangun_model_rnn(window_size, horizon):
    model = Sequential([
        Input(shape=(window_size, 1)),
        SimpleRNN(units=64, activation='tanh', return_sequences=True),
        Dropout(0.2),
        SimpleRNN(units=64, activation='tanh', return_sequences=False),
        Dropout(0.2),
        Dense(units=32, activation='relu'),
        Dense(units=horizon)  # Output langsung memproyeksikan vektor horizon multi-hari/multi-candle
    ])
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer, loss='mean_squared_error')
    return model


def inverse_2d(scaler, arr_2d):
    return scaler.inverse_transform(arr_2d.reshape(-1, 1)).reshape(arr_2d.shape)


def hitung_mape_aman(y_true, y_pred):
    mask = np.abs(y_true) > 1e-6
    if not np.any(mask):
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

print("Pipeline Preprocessing, Transformasi Skala & Arsitektur RNN Multi-Output Siap!")

# 5. Training & Prediksi Mode Multi-Hari (H+1, H+7, H+14, H+21, H+30, H+31)
Mendukung opsi **Model Caching** (`RETRAIN_MODEL = False`), menghasilkan sinyal trading (**STRONG BUY / BUY / WAIT / SELL**),
menentukan Target Price (TP), Stop Loss (SL), serta Risk-to-Reward Ratio (RRR).

In [ ]:
hasil_multihari = []
kurva_multihari = {}
skip_multihari = []

total = len(DAFTAR_SAHAM)
waktu_run = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')

print(f"=== MEMULAI ANALISIS MULTI-HARI ({total} Emiten) ===")

for idx, nama in enumerate(DAFTAR_SAHAM.keys(), start=1):
    sektor, kode = nama.split("_", 1)
    file_csv = os.path.join(DATA_DAILY_DIR, f"{nama}_1d.csv")
    model_path = os.path.join(MODEL_DIR, f"rnn_daily31_{nama}.keras")

    try:
        ds = siapkan_dataset(file_csv, WINDOW_SIZE, MAX_HORIZON_HARI)

        # Pengecekan Cache Model
        if (not RETRAIN_MODEL) and os.path.exists(model_path):
            model = load_model(model_path)
            mode_model_str = "LOAD_CACHED"
        else:
            model = bangun_model_rnn(WINDOW_SIZE, MAX_HORIZON_HARI)
            early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
            reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)

            model.fit(
                ds["X_train"], ds["y_train"],
                validation_split=0.1,
                epochs=EPOCHS,
                batch_size=BATCH_SIZE_DAILY,
                callbacks=[early_stop, reduce_lr],
                verbose=0
            )
            model.save(model_path)
            mode_model_str = "TRAINED"

        # Evaluasi pada Out-of-Sample Test Set
        pred_test_rp = inverse_2d(ds["scaler"], model.predict(ds["X_test"], verbose=0))
        y_test_rp = inverse_2d(ds["scaler"], ds["y_test"])

        rmse_h1 = float(np.sqrt(np.mean((pred_test_rp[:, 0] - y_test_rp[:, 0]) ** 2)))
        mape_h1 = hitung_mape_aman(y_test_rp[:, 0], pred_test_rp[:, 0])
        rmse_h31 = float(np.sqrt(np.mean((pred_test_rp[:, 30] - y_test_rp[:, 30]) ** 2)))
        mape_h31 = hitung_mape_aman(y_test_rp[:, 30], pred_test_rp[:, 30])

        # Proyeksi 31 Hari ke Depan dari Titik Penutupan Terakhir
        pred_31_rp = inverse_2d(ds["scaler"], model.predict(ds["X_future"], verbose=0)).flatten()
        close_last = ds["harga_terakhir"]

        # Hitung Persentase Potensi untuk Setiap Target Hari
        potensi_dict = {}
        for h in TARGET_HARI:
            potensi_dict[h] = ((pred_31_rp[h - 1] - close_last) / close_last) * 100

        potensi_h1 = potensi_dict[1]
        potensi_h7 = potensi_dict[7]
        potensi_h31 = potensi_dict[31]

        # Logika Penentuan Sinyal Trading & Risk Management
        if potensi_h31 >= 8.0 and potensi_h7 > 0 and mape_h31 <= 15.0:
            sinyal = "STRONG BUY"
        elif potensi_h31 >= 3.5 and mape_h31 <= 20.0:
            sinyal = "BUY"
        elif potensi_h31 <= -3.0:
            sinyal = "TAKE PROFIT / SELL"
        else:
            sinyal = "HOLD / WAIT"

        # Target Price (TP) & Stop Loss (SL)
        target_price = float(np.max(pred_31_rp))
        stop_loss = round(close_last * 0.95, 2)  # Batasan risiko 5% dari harga close terakhir
        potensi_gain_rp = max(target_price - close_last, 0)
        potensi_loss_rp = max(close_last - stop_loss, 1)
        rrr = round(potensi_gain_rp / potensi_loss_rp, 2)

        # Grade Akurasi Model
        if mape_h31 < 7.0:
            grade = "A (Sangat Presisi)"
        elif mape_h31 < 14.0:
            grade = "B (Presisi Baik)"
        elif mape_h31 < 22.0:
            grade = "C (Moderat)"
        else:
            grade = "D (Volatil Tinggi)"

        kurva_multihari[nama] = {
            "sektor": sektor,
            "kode": kode,
            "harga_terakhir": close_last,
            "waktu_terakhir": ds["waktu_terakhir"][:10],
            "prediksi": pred_31_rp,
            "potensi_h31": potensi_h31,
            "mape_h31": mape_h31,
            "sinyal": sinyal,
            "tp": target_price,
            "sl": stop_loss
        }

        baris = {
            "Waktu_Run": waktu_run,
            "Sektor": sektor,
            "Kode": kode,
            "Sektor_Emiten": nama,
            "Data_Terakhir": ds["waktu_terakhir"][:10],
            "Close_Terakhir": round(close_last, 2),
            "H+1": round(float(pred_31_rp[0]), 2),
            "H+7": round(float(pred_31_rp[6]), 2),
            "H+14": round(float(pred_31_rp[13]), 2),
            "H+21": round(float(pred_31_rp[20]), 2),
            "H+30": round(float(pred_31_rp[29]), 2),
            "H+31": round(float(pred_31_rp[30]), 2),
            "Potensi_H+1(%)": round(potensi_h1, 2),
            "Potensi_H+7(%)": round(potensi_h7, 2),
            "Potensi_H+31(%)": round(potensi_h31, 2),
            "Sinyal_Swing": sinyal,
            "Target_Price": round(target_price, 2),
            "Stop_Loss": stop_loss,
            "RRR": rrr,
            "RMSE_H+31": round(rmse_h31, 2),
            "MAPE_H+31(%)": round(mape_h31, 2),
            "Grade": grade
        }
        hasil_multihari.append(baris)

        print(f"[{idx:02d}/{total:02d}] {mode_model_str:11s} {nama:20s} | Close: {close_last:>7,.0f} -> H+31: {pred_31_rp[30]:>7,.0f} ({potensi_h31:+6.2f}%) | {sinyal:16s} | MAPE: {mape_h31:.2f}%")

    except Exception as e:
        skip_multihari.append({"Sektor_Emiten": nama, "Alasan": str(e)})
        print(f"[{idx:02d}/{total:02d}] SKIP {nama}: {e}")

    finally:
        tf.keras.backend.clear_session()
        gc.collect()

df_multihari = pd.DataFrame(hasil_multihari)
print(f"\n=== SELESAI MULTI-HARI: {len(df_multihari)} berhasil diproses, {len(skip_multihari)} di-skip ===")

if not df_multihari.empty:
    log_daily = os.path.join(LOG_DIR, 'riwayat_multihari.csv')
    latest_daily = os.path.join(LOG_DIR, 'latest_multihari.csv')
    df_multihari.to_csv(log_daily, mode='a', header=not os.path.exists(log_daily), index=False)
    df_multihari.to_csv(latest_daily, index=False)
    print("Log hasil multi-hari tersimpan di:", latest_daily)
    display(df_multihari[['Sektor', 'Kode', 'Close_Terakhir', 'H+1', 'H+7', 'H+14', 'H+21', 'H+30', 'H+31', 'Potensi_H+31(%)', 'Sinyal_Swing', 'Target_Price', 'Stop_Loss', 'RRR']].head(15))

# 6. Training & Prediksi Mode Scalping Intraday (Candle 5 Menit)
Memprediksi harga jangka sangat pendek (**+5m, +15m, +30m, +60m**), menghitung **Net Return setelah fee bursa (0.4%)**,
mendeteksi anomali volume, dan menyusun rencana eksekusi scalping (Entry Range, Scalp TP, Cut Loss SL).

In [ ]:
hasil_scalping = []
kurva_scalping = {}
skip_scalping = []

total = len(DAFTAR_SAHAM)
waktu_run_scalp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')

print(f"\n=== MEMULAI ANALISIS SCALPING 5 MENIT ({total} Emiten) ===")

for idx, nama in enumerate(DAFTAR_SAHAM.keys(), start=1):
    sektor, kode = nama.split("_", 1)
    file_csv = os.path.join(DATA_SCALP_DIR, f"{nama}_{SCALP_INTERVAL}.csv")
    model_path = os.path.join(MODEL_DIR, f"rnn_scalp5m_{nama}.keras")

    try:
        ds = siapkan_dataset(file_csv, WINDOW_SIZE, MAX_HORIZON_SCALP)

        if (not RETRAIN_MODEL) and os.path.exists(model_path):
            model = load_model(model_path)
            mode_model_str = "LOAD_CACHED"
        else:
            model = bangun_model_rnn(WINDOW_SIZE, MAX_HORIZON_SCALP)
            early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
            reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)

            model.fit(
                ds["X_train"], ds["y_train"],
                validation_split=0.1,
                epochs=EPOCHS,
                batch_size=BATCH_SIZE_SCALP,
                callbacks=[early_stop, reduce_lr],
                verbose=0
            )
            model.save(model_path)
            mode_model_str = "TRAINED"

        # Evaluasi Out-of-Sample Scalping
        pred_test_rp = inverse_2d(ds["scaler"], model.predict(ds["X_test"], verbose=0))
        y_test_rp = inverse_2d(ds["scaler"], ds["y_test"])

        rmse_5m = float(np.sqrt(np.mean((pred_test_rp[:, 0] - y_test_rp[:, 0]) ** 2)))
        mape_5m = hitung_mape_aman(y_test_rp[:, 0], pred_test_rp[:, 0])
        rmse_60m = float(np.sqrt(np.mean((pred_test_rp[:, 11] - y_test_rp[:, 11]) ** 2)))
        mape_60m = hitung_mape_aman(y_test_rp[:, 11], pred_test_rp[:, 11])

        # Proyeksi 12 Candle (60 Menit ke Depan)
        pred_scalp_rp = inverse_2d(ds["scaler"], model.predict(ds["X_future"], verbose=0)).flatten()
        close_last = ds["harga_terakhir"]

        delta_5m = ((pred_scalp_rp[0] - close_last) / close_last) * 100
        delta_15m = ((pred_scalp_rp[2] - close_last) / close_last) * 100
        delta_30m = ((pred_scalp_rp[5] - close_last) / close_last) * 100
        delta_60m = ((pred_scalp_rp[11] - close_last) / close_last) * 100
        net_gain_60m = delta_60m - FEE_THRESHOLD_PERSEN

        # Status Lonjakan Volume Intraday
        rasio_vol = ds.get("rasio_vol", 1.0)
        vol_surge_str = "SURGE (>1.3x)" if rasio_vol >= 1.3 else "NORMAL"

        # Logika Sinyal Scalping Khusus Day Trader
        if net_gain_60m >= 0.70 and mape_60m <= 2.5:
            sinyal_scalp = "STRONG SCALP BUY"
        elif net_gain_60m >= 0.35 and mape_60m <= 3.5:
            sinyal_scalp = "SPECULATIVE BUY"
        elif net_gain_60m <= -0.40:
            sinyal_scalp = "AVOID / SHORT-TERM BEARISH"
        else:
            sinyal_scalp = "SIDEWAYS (Belum Tutup Fee)"

        # Setup Plan Eksekusi Scalping
        entry_zone = f"{close_last * 0.998:,.0f} - {close_last:,.0f}"
        tp_scalp = round(max(pred_scalp_rp[0], pred_scalp_rp[2], pred_scalp_rp[11]), 2)
        sl_scalp = round(close_last * 0.99, 2)  # Disiplin ketat cut loss 1% untuk scalping

        kurva_scalping[nama] = {
            "sektor": sektor,
            "kode": kode,
            "harga_terakhir": close_last,
            "waktu_terakhir": ds["waktu_terakhir"],
            "prediksi": pred_scalp_rp,
            "delta_60m": delta_60m,
            "net_gain_60m": net_gain_60m,
            "mape_60m": mape_60m,
            "sinyal_scalp": sinyal_scalp,
            "tp": tp_scalp,
            "sl": sl_scalp
        }

        baris = {
            "Waktu_Run": waktu_run_scalp,
            "Sektor": sektor,
            "Kode": kode,
            "Sektor_Emiten": nama,
            "Candle_Terakhir": ds["waktu_terakhir"],
            "Last_Close": round(close_last, 2),
            "+5m": round(float(pred_scalp_rp[0]), 2),
            "+15m": round(float(pred_scalp_rp[2]), 2),
            "+30m": round(float(pred_scalp_rp[5]), 2),
            "+60m": round(float(pred_scalp_rp[11]), 2),
            "Delta_+5m(%)": round(delta_5m, 2),
            "Delta_+15m(%)": round(delta_15m, 2),
            "Delta_+30m(%)": round(delta_30m, 2),
            "Delta_+60m(%)": round(delta_60m, 2),
            "Net_Gain_60m(%)": round(net_gain_60m, 2),
            "Volume_Status": vol_surge_str,
            "Sinyal_Scalp": sinyal_scalp,
            "Entry_Zone": entry_zone,
            "TP_Scalp": tp_scalp,
            "SL_Scalp": sl_scalp,
            "MAPE_60m(%)": round(mape_60m, 2)
        }
        hasil_scalping.append(baris)

        print(f"[{idx:02d}/{total:02d}] {mode_model_str:11s} {nama:20s} | Close: {close_last:>7,.0f} -> +60m: {pred_scalp_rp[11]:>7,.0f} (Net: {net_gain_60m:+5.2f}%) | {sinyal_scalp:24s} | MAPE: {mape_60m:.2f}%")

    except Exception as e:
        skip_scalping.append({"Sektor_Emiten": nama, "Alasan": str(e)})
        print(f"[{idx:02d}/{total:02d}] SKIP {nama}: {e}")

    finally:
        tf.keras.backend.clear_session()
        gc.collect()

df_scalping = pd.DataFrame(hasil_scalping)
print(f"\n=== SELESAI SCALPING: {len(df_scalping)} berhasil diproses, {len(skip_scalping)} di-skip ===")

if not df_scalping.empty:
    log_scalp = os.path.join(LOG_DIR, 'riwayat_scalping.csv')
    latest_scalp = os.path.join(LOG_DIR, 'latest_scalping.csv')
    df_scalping.to_csv(log_scalp, mode='a', header=not os.path.exists(log_scalp), index=False)
    df_scalping.to_csv(latest_scalp, index=False)
    print("Log hasil scalping tersimpan di:", latest_scalp)
    display(df_scalping[['Sektor', 'Kode', 'Last_Close', '+5m', '+15m', '+30m', '+60m', 'Net_Gain_60m(%)', 'Volume_Status', 'Sinyal_Scalp', 'Entry_Zone', 'TP_Scalp', 'SL_Scalp']].head(15))

# 7. Visualisasi Grafik Dinamis & Auto-Save Laporan Gambar
Menghasilkan grafik proyeksi interaktif 4 emiten terbaik per sektor untuk **Multi-Hari** dan **Scalping Intraday**.
Grafik otomatis disimpan dalam format PNG beresolusi tinggi ke dalam folder `reports/`.

In [ ]:
def pilih_saham_untuk_plot(kurva_dict, df_sumber, kolom_skor, kolom_mape, batas_mape=25.0):
    if not kurva_dict or df_sumber.empty:
        return []

    # Saring berdasarkan MAPE yang wajar
    df_valid = df_sumber[df_sumber[kolom_mape] <= batas_mape]
    if df_valid.empty:
        df_valid = df_sumber

    terpilih = []
    # Ambil 1 saham dengan potensi tertinggi dari setiap sektor aktif
    for sektor in sektor_aktif:
        sub = df_valid[df_valid['Sektor'] == sektor].sort_values(by=kolom_skor, ascending=False)
        if not sub.empty:
            kandidat = sub.iloc[0]['Sektor_Emiten']
            if kandidat in kurva_dict:
                terpilih.append(kandidat)

    # Jika masih kurang dari 4 grafik, lengkapi dari peringkat teratas keseluruhan
    if len(terpilih) < 4:
        sisa = df_valid[~df_valid['Sektor_Emiten'].isin(terpilih)].sort_values(by=kolom_skor, ascending=False)
        for kandidat in sisa['Sektor_Emiten']:
            if kandidat in kurva_dict and len(terpilih) < 4:
                terpilih.append(kandidat)

    return terpilih[:4]


# ---------------------------------------------------------------------
# [1] PLOT PROYEKSI MULTI-HARI (H+1 s.d. H+31)
# ---------------------------------------------------------------------
plot_daily_list = pilih_saham_untuk_plot(kurva_multihari, df_multihari, 'Potensi_H+31(%)', 'MAPE_H+31(%)', batas_mape=25.0)
tgl_lap = datetime.datetime.now().strftime('%Y%m%d')

if plot_daily_list:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    sumbu_hari = np.arange(1, MAX_HORIZON_HARI + 1)

    for idx, ax in enumerate(axes):
        if idx >= len(plot_daily_list):
            ax.axis('off')
            continue

        nama = plot_daily_list[idx]
        d = kurva_multihari[nama]
        pred = d["prediksi"]

        # Garis Proyeksi RNN
        ax.plot(sumbu_hari, pred, color="#1f77b4", linewidth=2.5, label="Proyeksi RNN (H+1 - H+31)")
        # Garis Horizontal Harga Close Terakhir
        ax.axhline(d["harga_terakhir"], color="#555555", linestyle="--", linewidth=1.2, label=f"Close Terakhir (Rp {d['harga_terakhir']:,.0f})")
        # Garis Target Price & Stop Loss
        ax.axhline(d["tp"], color="#2ca02c", linestyle=":", linewidth=1.2, label=f"TP: Rp {d['tp']:,.0f}")
        ax.axhline(d["sl"], color="#d62728", linestyle=":", linewidth=1.2, label=f"SL: Rp {d['sl']:,.0f}")

        # Titik Target: H+1, H+7, H+14, H+21, H+30, H+31
        y_target = [pred[h - 1] for h in TARGET_HARI]
        ax.scatter(TARGET_HARI, y_target, color="#d62728", s=60, zorder=5, label="Target Hari")

        for h, val in zip(TARGET_HARI, y_target):
            chg = ((val - d["harga_terakhir"]) / d["harga_terakhir"]) * 100
            ax.annotate(f"H+{h}\n{val:,.0f}\n({chg:+.1f}%)", (h, val), textcoords="offset points", xytext=(0, 10), ha='center', fontsize=7.5, fontweight='bold')

        ax.set_title(f"{nama.replace('_', ' - ')} | Sinyal: {d['sinyal']} (MAPE: {d['mape_h31']:.1f}%)", fontsize=11, fontweight="bold")
        ax.set_xlabel("Hari Bursa Mendatang")
        ax.set_ylabel("Harga Saham (Rp)")
        ax.set_xticks(TARGET_HARI)
        ax.legend(loc="best", fontsize=7.5)
        ax.grid(True, alpha=0.3)

    plt.suptitle(f"Proyeksi Multi-Hari Saham BEI (H+1 s.d. H+31) - Top Pilihan per Sektor\nUpdate: {waktu_run}", fontsize=13, fontweight="bold", y=0.99)
    plt.tight_layout()
    path_gbr_daily = os.path.join(REPORT_DIR, f"grafik_multihari_{tgl_lap}.png")
    plt.savefig(path_gbr_daily, dpi=150, bbox_inches='tight')
    plt.show()
    print("Grafik Multi-Hari berhasil disimpan ke:", path_gbr_daily)
else:
    print("Tidak ada data grafik Multi-Hari yang memenuhi syarat untuk ditampilkan.")


# ---------------------------------------------------------------------
# [2] PLOT PROYEKSI SCALPING INTRADAY (5 Menit: +5m s.d. +60m)
# ---------------------------------------------------------------------
plot_scalp_list = pilih_saham_untuk_plot(kurva_scalping, df_scalping, 'Net_Gain_60m(%)', 'MAPE_60m(%)', batas_mape=4.0)

if plot_scalp_list:
    fig, axes = plt.subplots(2, 2, figsize=(16, 9))
    axes = axes.flatten()
    sumbu_menit = np.arange(1, MAX_HORIZON_SCALP + 1) * 5
    target_menit = [(i + 1) * 5 for i in TARGET_SCALP_INDEX]

    for idx, ax in enumerate(axes):
        if idx >= len(plot_scalp_list):
            ax.axis('off')
            continue

        nama = plot_scalp_list[idx]
        d = kurva_scalping[nama]
        pred = d["prediksi"]

        ax.plot(sumbu_menit, pred, color="#2ca02c", linewidth=2.5, label="Proyeksi Scalping (+5m s.d. +60m)")
        ax.axhline(d["harga_terakhir"], color="#555555", linestyle="--", linewidth=1.2, label=f"Last Close (Rp {d['harga_terakhir']:,.0f})")

        # Garis Breakeven Fee
        harga_fee = d["harga_terakhir"] * (1 + (FEE_THRESHOLD_PERSEN / 100))
        ax.axhline(harga_fee, color="#ff7f0e", linestyle=":", linewidth=1.2, label=f"Batas Fee 0.4% (Rp {harga_fee:,.0f})")

        y_target = [pred[i] for i in TARGET_SCALP_INDEX]
        ax.scatter(target_menit, y_target, color="#1f77b4", s=60, zorder=5, label="Titik Target Scalp")

        for m, val in zip(target_menit, y_target):
            chg = ((val - d["harga_terakhir"]) / d["harga_terakhir"]) * 100
            ax.annotate(f"+{m}m\n{val:,.0f}\n({chg:+.2f}%)", (m, val), textcoords="offset points", xytext=(0, 10), ha='center', fontsize=7.5, fontweight='bold')

        ax.set_title(f"{nama.replace('_', ' - ')} | Sinyal: {d['sinyal_scalp']} (MAPE: {d['mape_60m']:.2f}%)", fontsize=11, fontweight="bold")
        ax.set_xlabel("Menit Intraday Mendatang")
        ax.set_ylabel("Harga Saham (Rp)")
        ax.set_xticks(target_menit)
        ax.legend(loc="best", fontsize=7.5)
        ax.grid(True, alpha=0.3)

    plt.suptitle(f"Proyeksi Scalping Intraday 5 Menit (+5m s.d. +60m) - Top Pilihan per Sektor\nUpdate: {waktu_run_scalp}", fontsize=13, fontweight="bold", y=0.99)
    plt.tight_layout()
    path_gbr_scalp = os.path.join(REPORT_DIR, f"grafik_scalping_{tgl_lap}.png")
    plt.savefig(path_gbr_scalp, dpi=150, bbox_inches='tight')
    plt.show()
    print("Grafik Scalping berhasil disimpan ke:", path_gbr_scalp)
else:
    print("Tidak ada data grafik Scalping yang memenuhi syarat untuk ditampilkan.")

# 8. Screener Top 10, Ringkasan Sektoral & Panduan Trading Harian
Menampilkan rangkuman eksekutif saham terbaik hari ini beserta aturan money management dan ekspor tabel rekomendasi.

In [ ]:
print("=" * 75)
print("=== [1] TOP 10 SWING TRADING (POTENSI H+31 TERTINGGI DENGAN MAPE <= 18%) ===")
print("=" * 75)
if not df_multihari.empty:
    top_swing = df_multihari[df_multihari['MAPE_H+31(%)'] <= 18.0].sort_values(by='Potensi_H+31(%)', ascending=False).head(10)
    kolom_swing_tampil = [
        'Sektor', 'Kode', 'Close_Terakhir', 'H+1', 'H+7', 'H+14', 'H+21', 'H+30', 'H+31',
        'Potensi_H+31(%)', 'Sinyal_Swing', 'Target_Price', 'Stop_Loss', 'RRR', 'Grade'
    ]
    display(top_swing[kolom_swing_tampil])
else:
    print("Data hasil multi-hari belum tersedia.")

print("\n" + "=" * 75)
print("=== [2] TOP 10 SCALPING 5M (POTENSI NET GAIN SETELAH FEE & MAPE <= 3%) ===")
print("=" * 75)
if not df_scalping.empty:
    top_scalp = df_scalping[
        (df_scalping['Net_Gain_60m(%)'] > 0.3) &
        (df_scalping['MAPE_60m(%)'] <= 3.0)
    ].sort_values(by='Net_Gain_60m(%)', ascending=False).head(10)

    if top_scalp.empty:
        # Jika pasar sedang lesu, tampilkan kandidat dengan net gain tertinggi tanpa filter ketat
        top_scalp = df_scalping.sort_values(by='Net_Gain_60m(%)', ascending=False).head(10)

    kolom_scalp_tampil = [
        'Sektor', 'Kode', 'Last_Close', '+5m', '+15m', '+30m', '+60m',
        'Net_Gain_60m(%)', 'Volume_Status', 'Sinyal_Scalp', 'Entry_Zone', 'TP_Scalp', 'SL_Scalp', 'MAPE_60m(%)'
    ]
    display(top_scalp[kolom_scalp_tampil])
else:
    print("Data hasil scalping belum tersedia.")

print("\n" + "=" * 75)
print("=== [3] RANGKUMAN KINERJA PER SEKTOR (SWING MULTI-HARI) ===")
print("=" * 75)
if not df_multihari.empty:
    ringkasan_sektor = df_multihari.groupby('Sektor').agg(
        Jumlah_Emiten=('Kode', 'count'),
        Rata2_Potensi_H1_Persen=('Potensi_H+1(%)', 'mean'),
        Rata2_Potensi_H7_Persen=('Potensi_H+7(%)', 'mean'),
        Rata2_Potensi_H31_Persen=('Potensi_H+31(%)', 'mean'),
        Rata2_MAPE_H31=('MAPE_H+31(%)', 'mean')
    ).round(2)
    display(ringkasan_sektor)

print("\n" + "=" * 75)
print("=== [4] PANDUAN TRADING HARIAN & MONEY MANAGEMENT (EVERYDAY USE PLAYBOOK) ===")
print("=" * 75)
print("""
[PANDUAN EKSEKUSI HARIAN]:
1. Jam Perdagangan BEI (WIB):
   - Sesi I : 09:00 - 12:00 (Jumat: 09:00 - 11:30)
   - Sesi II: 13:30 - 15:49 (Jumat: 14:00 - 15:49)
   - Waktu terbaik untuk Scalping Intraday: 09:05 - 10:00 (Volatilitas Pembukaan) dan 14:30 - 15:30 (Penutupan).

2. Disiplin Stop Loss (SL):
   - Scalping  : Maksimal cut loss 1.0% s/d 1.5% dari harga beli.
   - Swing     : Maksimal cut loss 3.0% s/d 5.0% atau ikuti batas level Stop_Loss pada tabel.

3. Aturan Alokasi Modal (Money Management):
   - Jangan alokasikan lebih dari 20% total portofolio pada satu saham tunggal.
   - Risiko per trade (Risk per Trade) tidak boleh melebihi 1% - 2% dari total modal akun Anda.

4. Filter Konfirmasi Teknikal:
   - Prioritaskan saham dengan status Volume 'SURGE (>1.3x)' pada scalping.
   - Pilih emiten dengan Grade Akurasi Model 'A' atau 'B' untuk kepercayaan proyeksi yang lebih tinggi.
""")

# Ekspor Rekomendasi Hari Ini ke File CSV untuk Pemantauan di Excel / Mobile
if not df_multihari.empty:
    rekomendasi_swing_path = os.path.join(REPORT_DIR, f"rekomendasi_swing_{tgl_lap}.csv")
    df_multihari.to_csv(rekomendasi_swing_path, index=False)
    print(f"File CSV Rekomendasi Swing tersimpan di: {rekomendasi_swing_path}")

if not df_scalping.empty:
    rekomendasi_scalp_path = os.path.join(REPORT_DIR, f"rekomendasi_scalp_{tgl_lap}.csv")
    df_scalping.to_csv(rekomendasi_scalp_path, index=False)
    print(f"File CSV Rekomendasi Scalping tersimpan di: {rekomendasi_scalp_path}")
print("=" * 75)